In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [2]:
df = pd.read_excel("../dataset/student_performance_dataset_enhanced.xlsx")

In [3]:
X = df.drop("Final_Grade", axis=1)
y = df["Final_Grade"]

In [4]:
preprocessor = joblib.load("../Model/preprocessor.pkl")
label_encoder = joblib.load("../Model/label_encoder.pkl")

y = label_encoder.transform(y)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [6]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5)
}

In [7]:
results = []

best_model = None
best_accuracy = 0

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted")
    recall = recall_score(y_test, y_pred, average="weighted")
    f1 = f1_score(y_test, y_pred, average="weighted")

    results.append([
        name,
        accuracy,
        precision,
        recall,
        f1
    ])

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_model = pipeline

In [8]:
results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ]
)

results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
)

print(results_df)

                 Model  Accuracy  Precision  Recall  F1 Score
0  Logistic Regression    0.9300   0.931517  0.9300  0.930582
2        Random Forest    0.9175   0.918924  0.9175  0.917013
3  K-Nearest Neighbors    0.8855   0.882838  0.8855  0.883930
1        Decision Tree    0.8815   0.884103  0.8815  0.882707


In [9]:
joblib.dump(best_model, "../Model/best_model.pkl")

['../Model/best_model.pkl']

In [11]:
results_df.to_csv("../Report/model_results.csv", index=False)